| 上下文管理器                              | 位置                              | 用途                    | 关键场景                           |
| ----------------------------------- | ------------------------------- | --------------------- | ------------------------------ |
| `train_mode()`                      | `workers/engine/base.py`        | 训练模式切换 + 显存管理         | `train_mini_batch()` 调用        |
| `eval_mode()`                       | `workers/engine/base.py`        | 评估模式切换 + 显存管理         | `infer_batch()` / `generate()` |
| `get_init_weight_context_manager()` | `utils/fsdp_utils.py`           | 模型初始化设备控制             | `from_pretrained()` 时避免 OOM    |
| `merged_lora_context()`             | `utils/fsdp_utils.py`           | LoRA 临时合并/恢复          | vLLM/SGLang 权重同步               |
| `marked_timer()`                    | `utils/profiler/performance.py` | 带标记的代码块计时             | 训练各阶段性能统计                      |
| `simple_timer()`                    | `utils/profiler/performance.py` | 纯软件计时                 | 数据加载等轻量计时                      |
| `maybe_patch_fsdp_module()`         | `utils/fsdp_utils.py`           | FSDP 模块兼容性修补          | FSDP2 特殊模型适配                   |
| `MegatronVLLMShardingManager`       | `workers/megatron_workers.py`   | Megatron ↔ vLLM 权重重分片 | HybridEngine 场景                |


# Engine 模式管理上下文管理器

##  train_mode() / eval_mode()

- 位置：verl/workers/engine/base.py（基类）

<font color='red'>这是 verl Engine 最核心的上下文管理器，所有后端（FSDP / Megatron / SGLang）都通过继承 BaseEngineCtx 实现。它们负责在训练/推理阶段自动切换模型模式、管理显存和梯度状态。</font>

In [ ]:
from verl.workers.engine.base import BaseEngine

# 训练模式：自动加载模型到 GPU、启用梯度、设为 train 模式
with engine.train_mode():
    loss = engine.train_batch(data)

# 评估模式：自动加载模型到 GPU、禁用梯度、设为 eval 模式
with engine.eval_mode():
    output = engine.infer_batch(data)

### BaseEngineCtx 实现逻辑(base.py:235-302）

| 阶段          | 操作                                                                                                 |
| ----------- | -------------------------------------------------------------------------------------------------- |
| `__enter__` | 1. 若启用 offload：将模型从 CPU 加载到 GPU<br>2. 将模型设为 `train()` 或 `eval()` 模式<br>3. 管理梯度上下文（`requires_grad`） |
| `__exit__`  | 1. 若启用 offload：将模型从 GPU 卸载回 CPU<br>2. 恢复原始模式状态                                                     |

#### FSDP 实现（fsdp/transformer_impl.py:453-467）

In [ ]:
class FSDPEngineCtx(BaseEngineCtx):
    def __enter__(self):
        if self.engine._is_offload_param:
            load_fsdp_model_to_gpu(self.engine.module)  # CPU → GPU
        self.engine.module.train(self.is_training)
        return self
    
    def __exit__(self, exc_type, exc_value, traceback):
        if self.engine._is_offload_param:
            offload_fsdp_model_to_cpu(self.engine.module)  # GPU → CPU
        super().__exit__(exc_type, exc_value, traceback)

#### Megatron 实现（megatron/engine_impl.py:468-512）

In [ ]:
# Train mode entry
if self.engine._is_offload_param:
    load_megatron_model_to_gpu(self.engine.module, load_grad=True)
if self.engine._is_offload_optimizer:
    load_megatron_optimizer(optimizer=self.engine.optimizer)
    

# 模型初始化上下文管理器

##  get_init_weight_context_manager()

- 位置：verl/utils/fsdp_utils.py（行 85-98）

<font color='red'>用于控制模型权重初始化时的设备上下文，支持延迟初始化（meta tensor）以节省显存。</font>

### 逻辑：
- use_meta_tensor=True + rank ≠ 0：使用 accelerate.init_empty_weights()（参数在 meta device 上，不占显存）
- use_meta_tensor=True + rank = 0：在 CPU 上初始化真实权重
- use_meta_tensor=False：直接在 CPU 上初始化

<font color='red'>这是 verl 处理大模型加载时避免 OOM 的关键机制</font>

In [ ]:
from verl.utils.fsdp_utils import get_init_weight_context_manager

init_context = get_init_weight_context_manager(
    use_meta_tensor=True,      # 使用 meta tensor 延迟初始化
    mesh=self.device_mesh      # 设备网格
)

with init_context():
    model = AutoModelForCausalLM.from_pretrained(...)

# LoRA 相关上下文管理器

##  merged_lora_context()

- 位置：verl/utils/fsdp_utils.py（行 230+）

<font color='red'>临时将 LoRA adapter 合并到基模型权重中，执行操作后再恢复。用于 vLLM/SGLang 权重同步等场景。</font>

| 阶段          | 操作                                                                                                          |
| ----------- | ----------------------------------------------------------------------------------------------------------- |
| `__enter__` | 1. 若 `backup_adapters=True`：先备份基模型权重（禁用 LoRA 后）<br>2. 调用 `fsdp_merge_unmerge(actor, do_merge=True)` 合并 LoRA |
| `__exit__`  | 1. 若备份存在：从 CPU 备份恢复基模型权重（数值更稳定）<br>2. 否则：调用 `fsdp_merge_unmerge(actor, do_merge=False)` 取消合并                |

<font color='red'>为什么需要 backup 模式：直接 unmerge 可能引入数值误差，而备份-恢复方式更稳定</font>

In [ ]:
from verl.utils.fsdp_utils import merged_lora_context

with merged_lora_context(actor, backup_adapters=True):
    # 此时 LoRA 已合并到基模型
    sync_weights_to_vllm(actor)   # 同步到 rollout engine
    # 退出时自动恢复

# 性能分析上下文管理器
##  marked_timer()
- 位置：verl/utils/profiler/performance.py（行 78+）

<font color='red'>带标记的计时上下文管理器，支持 NVTX 等硬件 profiler 标记。</font>

In [ ]:
from verl.utils.profiler.performance import marked_timer

timing_raw = {}

with marked_timer("actor_forward", timing_raw, color="blue", domain="actor"):
    logits = actor_model(input_ids)
    
# timing_raw["actor_forward"] 自动累加耗时

## simple_timer()

- 位置：verl/utils/profiler/performance.py（行 68+）

<font color='red'>基础计时器，不带 NVTX 标记，用于纯软件计时。</font>

In [ ]:
from verl.utils.profiler.performance import simple_timer

with simple_timer("data_loading", timing_raw):
    batch = next(dataloader)

## DistProfiler.start() / stop()
- 位置：verl/utils/profiler/profile.py 
<font color='red'>分布式性能分析器的生命周期管理，内部使用 with 语句包装。</font>

In [ ]:
profiler = DistProfiler(config)

with profiler:
    # 自动调用 start() 和 stop()
    train_step()

# FSDP 辅助上下文管理器
##  maybe_patch_fsdp_module()
- 位置：verl/utils/fsdp_utils.py 
<font color='red'>临时修补 FSDP 模块的 ABC 继承关系，用于兼容某些特殊模型结构。</font>

In [ ]:
from verl.utils.fsdp_utils import maybe_patch_fsdp_module

with maybe_patch_fsdp_module(model):
    # 在此上下文中，FSDPModule 被临时替换为兼容版本
    apply_fsdp2(model, ...)

# Megatron 相关上下文管理器
##  MegatronVLLMShardingManager
- 位置：verl/workers/megatron_workers.py 
<font color='red'>用于在 Actor 和 Rollout 之间执行实际的权重 resharding 的上下文管理器。</font>


In [ ]:
with MegatronVLLMShardingManager(actor_weights, rollout_config) as sharding_manager:
    # 自动完成 TP/PP 维度的权重重新分片
    rollout_output = rollout_engine.generate(...)

# 典型调用链示例
在 verl 的一个训练步骤中，上下文管理器通常这样嵌套使用（从堆栈跟踪可见）

In [ ]:
with engine.train_mode():                    # 1. 加载模型到 GPU，设 train 模式
    with marked_timer("actor_update", timing):
        with merged_lora_context(actor):     # 2. 临时合并 LoRA（如需要）
            loss = actor.train_batch(data)   # 3. 实际训练
# 退出时自动：恢复 LoRA → 停止计时 → 卸载模型到 CPU